# 05 · Interpretabilidad y análisis de errores del modelo final

Este notebook interpreta el modelo final **XGBoost** y lo compara con la línea base de **regresión logística**. Genera importancia global de variables y analiza los errores sobre el conjunto test por estación, hora y clase de riesgo.

La elección de XGBoost ya se cerró mediante validation en el notebook 04. Por ello, aquí el test se usa exclusivamente para describir el comportamiento final y no para ajustar hiperparámetros ni seleccionar modelos.

## Reproducibilidad y dependencias

Para reproducir los resultados publicados, XGBoost usa una muestra de 500.000 filas de train y la regresión logística 750.000, igual que en los notebooks 04 y 02. Cada imputador y codificador se ajusta solo con su muestra de train; el test nunca participa en esos ajustes.

In [ ]:

# %pip install scikit-learn
# %pip install xgboost

from pathlib import Path
import gc

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score, classification_report, confusion_matrix, f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# Encontramos la raíz del proyecto tanto desde la carpeta raíz como desde notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'Datos modelado').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURES_DIR = PROJECT_ROOT / 'Datos modelado' / 'estacion_hora_features'
OUTPUT_DIR = PROJECT_ROOT / 'Datos modelado' / 'interpretabilidad'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'risk_class_1h'
RANDOM_STATE = 42
CHUNK_SIZE = 100_000
MAX_TRAIN_ROWS_XGBOOST = 500_000
MAX_TRAIN_ROWS_LOGISTIC = 750_000
CLASS_LABELS = {0: 'estable', 1: 'vaciado', 2: 'saturación'}


## Variables y carga de datos

Se emplean exactamente las variables disponibles en el instante de predicción. Además de esas variables, se conservan identificadores de estación y tiempo solo para agrupar errores posteriormente; no se incorporan como nuevas variables predictivas.

In [ ]:
NUMERIC_FEATURES = [
    'capacity', 'bikes_available', 'docks_available', 'reservations_count',
    'occupancy_ratio', 'light', 'weather_available',
    'uv_radiation_median_mw_m2', 'wind_speed_median_m_s',
    'wind_direction_sin_mean', 'wind_direction_cos_mean',
    'temperature_median_c', 'relative_humidity_median_pct',
    'barometric_pressure_median_mb', 'solar_radiation_median_w_m2',
    'precipitation_mean_l_m2', 'precipitation_max_l_m2',
    'n_temperature', 'n_relative_humidity', 'n_precipitation',
    'hour', 'day_of_week', 'month', 'week_of_year',
    'occupancy_ratio_lag_1h', 'occupancy_ratio_lag_2h', 'occupancy_ratio_lag_24h',
    'bikes_available_lag_1h', 'bikes_available_lag_2h', 'bikes_available_lag_24h',
    'net_flow_lag_1h', 'net_flow_lag_2h', 'net_flow_lag_24h',
    'departures_count_lag_1h', 'departures_count_lag_2h', 'departures_count_lag_24h',
    'arrivals_count_lag_1h', 'arrivals_count_lag_2h', 'arrivals_count_lag_24h',
    'occupancy_ratio_mean_previous_3h', 'occupancy_ratio_mean_previous_24h',
    'net_flow_mean_previous_3h', 'net_flow_mean_previous_24h',
    'departures_count_mean_previous_3h', 'departures_count_mean_previous_24h',
    'arrivals_count_mean_previous_3h', 'arrivals_count_mean_previous_24h',
]
CATEGORICAL_FEATURES = ['station_id', 'tipo_dia']

# Estas columnas se guardan para localizar patrones de error; no todas alimentan al modelo.
METADATA_COLUMNS = ['fecha_hora_local', 'station_id', 'station_number', 'station_name', 'hour', 'tipo_dia']
READ_COLUMNS = list(dict.fromkeys(NUMERIC_FEATURES + CATEGORICAL_FEATURES + METADATA_COLUMNS + [TARGET, 'dataset_split']))

feature_files = sorted(FEATURES_DIR.glob('estacion_hora_features_*.csv'))
if len(feature_files) != 48:
    raise FileNotFoundError(f'Se esperaban 48 CSV mensuales y se encontraron {len(feature_files)} en {FEATURES_DIR}')

def count_train_rows() -> int:
    """Cuenta los casos etiquetados de train sin cargar todas las columnas."""
    total = 0
    for file_path in feature_files:
        for chunk in pd.read_csv(file_path, usecols=[TARGET, 'dataset_split'], chunksize=CHUNK_SIZE, low_memory=False):
            total += int((chunk['dataset_split'].eq('train') & chunk[TARGET].notna()).sum())
    return total

TRAIN_TOTAL = count_train_rows()
print(f'Filas etiquetadas disponibles en train: {TRAIN_TOTAL:,}')


In [ ]:
def load_test_metadata() -> pd.DataFrame:
    """Carga todas las filas etiquetadas del test, con metadatos para el análisis posterior."""
    parts = []
    for file_path in feature_files:
        for chunk in pd.read_csv(file_path, usecols=READ_COLUMNS, chunksize=CHUNK_SIZE, low_memory=False):
            keep = chunk['dataset_split'].eq('test') & chunk[TARGET].notna()
            if keep.any():
                parts.append(chunk.loc[keep].copy())
    test_frame = pd.concat(parts, ignore_index=True)
    test_frame[TARGET] = test_frame[TARGET].astype('int8')
    return test_frame

def load_train_for_xgboost(max_rows: int) -> pd.DataFrame:
    """Replica el muestreo aleatorio del notebook 04 para el modelo XGBoost final."""
    fraction = min(1.0, max_rows / TRAIN_TOTAL)
    rng = np.random.default_rng(RANDOM_STATE)
    parts = []
    for file_path in feature_files:
        for chunk in pd.read_csv(file_path, usecols=NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET, 'dataset_split'], chunksize=CHUNK_SIZE, low_memory=False):
            train_chunk = chunk.loc[chunk['dataset_split'].eq('train') & chunk[TARGET].notna()].copy()
            if fraction < 1:
                # Se genera una selección reproducible usando exclusivamente filas de train.
                train_chunk = train_chunk.loc[rng.random(len(train_chunk)) < fraction]
            if not train_chunk.empty:
                parts.append(train_chunk)
    train_frame = pd.concat(parts, ignore_index=True)
    train_frame[TARGET] = train_frame[TARGET].astype('int8')
    return train_frame

def load_train_for_logistic(max_rows: int) -> pd.DataFrame:
    """Replica el muestreo estratificado por bloque empleado en el notebook 02."""
    fraction = min(1.0, max_rows / TRAIN_TOTAL)
    parts = []
    for file_path in feature_files:
        for chunk in pd.read_csv(file_path, usecols=NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET, 'dataset_split'], chunksize=200_000, low_memory=False):
            train_chunk = chunk.loc[chunk['dataset_split'].eq('train') & chunk[TARGET].notna()].copy()
            if fraction < 1 and not train_chunk.empty:
                # El muestreo se conserva por clase para mantener la distribución de la línea base.
                train_chunk = train_chunk.groupby(TARGET, group_keys=False).sample(frac=fraction, random_state=RANDOM_STATE)
            if not train_chunk.empty:
                parts.append(train_chunk)
    train_frame = pd.concat(parts, ignore_index=True)
    train_frame[TARGET] = train_frame[TARGET].astype('int8')
    return train_frame

test = load_test_metadata()
print(f'Filas etiquetadas en test: {len(test):,}')
print(test[TARGET].value_counts(normalize=True).sort_index().rename(index=CLASS_LABELS))


## Entrenamiento reproducible, sin fuga, y predicciones finales

Esta sección no ajusta modelos mediante test: reconstruye los dos modelos con los parámetros ya fijados y obtiene predicciones sobre el test. Cada preprocesador se ajusta exclusivamente con train porque los tamaños de muestra originales no son iguales.

In [ ]:
def make_preprocessor() -> ColumnTransformer:
    """Crea un preprocesador nuevo que se ajustará solo al train del modelo correspondiente."""
    numeric_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
    ])
    categorical_pipeline = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('one_hot', OneHotEncoder(handle_unknown='ignore')),
    ])
    return ColumnTransformer(transformers=[
        ('numeric', numeric_pipeline, NUMERIC_FEATURES),
        ('categorical', categorical_pipeline, CATEGORICAL_FEATURES),
    ])

def test_metrics(y_true: pd.Series, y_pred: np.ndarray, name: str) -> dict:
    """Resume las dos métricas fijadas para comparar el modelo final y la línea base."""
    return {
        'model': name,
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'f1_macro': f1_score(y_true, y_pred, average='macro'),
    }

# --- XGBoost final: mismo tamaño de train y mismos hiperparámetros que en el notebook 04. ---
train_xgb = load_train_for_xgboost(MAX_TRAIN_ROWS_XGBOOST)
xgb_preprocessor = make_preprocessor()
X_train_xgb = xgb_preprocessor.fit_transform(train_xgb[NUMERIC_FEATURES + CATEGORICAL_FEATURES])
X_test_xgb = xgb_preprocessor.transform(test[NUMERIC_FEATURES + CATEGORICAL_FEATURES])

xgb_model = XGBClassifier(
    objective='multi:softprob', num_class=3, eval_metric='mlogloss',
    n_estimators=350, max_depth=8, learning_rate=0.08,
    subsample=0.8, colsample_bytree=0.8,
    tree_method='hist', n_jobs=-1, random_state=RANDOM_STATE,
)
xgb_model.fit(X_train_xgb, train_xgb[TARGET])
prediction_xgb = xgb_model.predict(X_test_xgb).astype('int8')

# Liberamos las matrices grandes de XGBoost antes de entrenar la línea base.
del train_xgb, X_train_xgb
gc.collect()

# --- Regresión logística: mismo tamaño de train y parámetros que en el notebook 02. ---
train_logistic = load_train_for_logistic(MAX_TRAIN_ROWS_LOGISTIC)
logistic_preprocessor = make_preprocessor()
X_train_logistic = logistic_preprocessor.fit_transform(train_logistic[NUMERIC_FEATURES + CATEGORICAL_FEATURES])
X_test_logistic = logistic_preprocessor.transform(test[NUMERIC_FEATURES + CATEGORICAL_FEATURES])

logistic_model = LogisticRegression(
    solver='saga', max_iter=300, multi_class='multinomial',
    n_jobs=-1, random_state=RANDOM_STATE,
)
logistic_weights = compute_sample_weight(class_weight='balanced', y=train_logistic[TARGET])
logistic_model.fit(X_train_logistic, train_logistic[TARGET], sample_weight=logistic_weights)
prediction_logistic = logistic_model.predict(X_test_logistic).astype('int8')

results_test = pd.DataFrame([
    test_metrics(test[TARGET], prediction_xgb, 'XGBoost final'),
    test_metrics(test[TARGET], prediction_logistic, 'Regresión logística (línea base)'),
]).sort_values('f1_macro', ascending=False)
results_test.to_csv(OUTPUT_DIR / 'comparacion_modelo_final_y_linea_base_test.csv', index=False, encoding='utf-8-sig')
results_test


## Importancia global de variables en XGBoost

La importancia por ganancia muestra cuánto ayudan las variables a reducir el error en los árboles. No demuestra causalidad. Se muestran tanto las variables transformadas como una agregación por variable original, necesaria porque `station_id` y `tipo_dia` se codifican en varias columnas.

In [ ]:
transformed_feature_names = xgb_preprocessor.get_feature_names_out()
importance_transformed = pd.DataFrame({
    'variable_transformada': transformed_feature_names,
    'importance_gain': xgb_model.feature_importances_,
}).sort_values('importance_gain', ascending=False)

def original_feature_name(transformed_name: str) -> str:
    """Agrupa indicadores de ausencia y categorías codificadas bajo su variable de origen."""
    name = transformed_name.split('__', maxsplit=1)[-1]
    if name.startswith('missingindicator_'):
        return 'indicadores_de_ausencia'
    for categorical_name in CATEGORICAL_FEATURES:
        if name == categorical_name or name.startswith(f'{categorical_name}_'):
            return categorical_name
    return name

importance_transformed['variable_origen'] = importance_transformed['variable_transformada'].map(original_feature_name)
importance_global = (
    importance_transformed.groupby('variable_origen', as_index=False)['importance_gain'].sum()
    .sort_values('importance_gain', ascending=False)
)
importance_transformed.to_csv(OUTPUT_DIR / 'importancia_xgboost_transformada.csv', index=False, encoding='utf-8-sig')
importance_global.to_csv(OUTPUT_DIR / 'importancia_xgboost_global.csv', index=False, encoding='utf-8-sig')

top_importance = importance_global.head(20).sort_values('importance_gain')
ax = top_importance.plot.barh(x='variable_origen', y='importance_gain', legend=False, figsize=(9, 7), color='#0B6E99')
ax.set_title('Top 20 variables globales según XGBoost')
ax.set_xlabel('Importancia relativa por ganancia')
ax.set_ylabel('Variable')
plt.tight_layout()
plt.show()
importance_global.head(20)


## Predicciones y métricas por clase

El archivo de predicciones conserva una fila por observación de test y modelo. Esto permite auditar cada error. Los resúmenes por clase separan el rendimiento en estabilidad, vaciado y saturación; son esenciales porque el F1 macro da el mismo peso a las tres clases.

In [ ]:
def prediction_table(model_name: str, prediction: np.ndarray) -> pd.DataFrame:
    """Construye una tabla auditable de predicciones sin modificar los datos de origen."""
    table = test[METADATA_COLUMNS + [TARGET]].copy()
    table['model'] = model_name
    table['predicted_risk_class_1h'] = prediction
    table['real_class_name'] = table[TARGET].map(CLASS_LABELS)
    table['predicted_class_name'] = table['predicted_risk_class_1h'].map(CLASS_LABELS)
    table['is_error'] = table[TARGET].ne(table['predicted_risk_class_1h'])
    table['error_pair'] = table['real_class_name'] + ' → ' + table['predicted_class_name']
    return table

predictions = pd.concat([
    prediction_table('XGBoost final', prediction_xgb),
    prediction_table('Regresión logística (línea base)', prediction_logistic),
], ignore_index=True)
predictions.to_csv(OUTPUT_DIR / 'predicciones_test_modelo_final_y_linea_base.csv', index=False, encoding='utf-8-sig')

def class_summary(frame: pd.DataFrame) -> pd.DataFrame:
    """Calcula soporte, recall y tasa de error para cada clase real y modelo."""
    summary = (
        frame.groupby(['model', TARGET, 'real_class_name'], as_index=False)
        .agg(observaciones=(TARGET, 'size'), errores=('is_error', 'sum'))
    )
    summary['aciertos'] = summary['observaciones'] - summary['errores']
    summary['recall_por_clase'] = summary['aciertos'] / summary['observaciones']
    summary['tasa_error'] = summary['errores'] / summary['observaciones']
    return summary.sort_values(['model', TARGET])

errors_by_real_class = class_summary(predictions)
errors_by_real_class.to_csv(OUTPUT_DIR / 'errores_por_clase_real.csv', index=False, encoding='utf-8-sig')
errors_by_real_class


In [ ]:
# Guardamos matrices de confusión en formato largo para poder compararlas o graficarlas después.
confusion_parts = []
reports_parts = []
for model_name, model_frame in predictions.groupby('model'):
    matrix = confusion_matrix(model_frame[TARGET], model_frame['predicted_risk_class_1h'], labels=[0, 1, 2])
    for real_class in range(3):
        for predicted_class in range(3):
            confusion_parts.append({
                'model': model_name,
                'real_class': CLASS_LABELS[real_class],
                'predicted_class': CLASS_LABELS[predicted_class],
                'n_observaciones': int(matrix[real_class, predicted_class]),
            })
    report = classification_report(
        model_frame[TARGET], model_frame['predicted_risk_class_1h'],
        labels=[0, 1, 2], target_names=[CLASS_LABELS[i] for i in range(3)],
        output_dict=True, zero_division=0,
    )
    report_frame = pd.DataFrame(report).T.reset_index(names='clase_o_metrica')
    report_frame.insert(0, 'model', model_name)
    reports_parts.append(report_frame)

confusion_long = pd.DataFrame(confusion_parts)
classification_reports = pd.concat(reports_parts, ignore_index=True)
confusion_long.to_csv(OUTPUT_DIR / 'matrices_confusion_test.csv', index=False, encoding='utf-8-sig')
classification_reports.to_csv(OUTPUT_DIR / 'metricas_por_clase_test.csv', index=False, encoding='utf-8-sig')
classification_reports


## Errores por estación y hora

La tasa de error se acompaña de un mínimo de observaciones para no sobrerreaccionar a estaciones u horas con pocos casos. Estos resultados identifican dónde conviene revisar calidad de datos, comportamiento operativo o posibles mejoras del modelo.

In [ ]:
def error_summary(frame: pd.DataFrame, group_columns: list[str]) -> pd.DataFrame:
    """Resume número de casos, errores y tasa de error para cualquier agrupación."""
    result = (
        frame.groupby(['model'] + group_columns, as_index=False)
        .agg(observaciones=('is_error', 'size'), errores=('is_error', 'sum'))
    )
    result['aciertos'] = result['observaciones'] - result['errores']
    result['tasa_error'] = result['errores'] / result['observaciones']
    return result

errors_by_station = error_summary(predictions, ['station_id', 'station_number', 'station_name'])
errors_by_hour = error_summary(predictions, ['hour'])

# Se excluyen grupos con menos de 100 observaciones del ranking para que sea estable y accionable.
stations_to_review = (
    errors_by_station.query('observaciones >= 100')
    .sort_values(['model', 'tasa_error', 'errores'], ascending=[True, False, False])
)

errors_by_station.to_csv(OUTPUT_DIR / 'errores_por_estacion.csv', index=False, encoding='utf-8-sig')
errors_by_hour.to_csv(OUTPUT_DIR / 'errores_por_hora.csv', index=False, encoding='utf-8-sig')
stations_to_review.to_csv(OUTPUT_DIR / 'estaciones_prioritarias_por_error.csv', index=False, encoding='utf-8-sig')

print('Estaciones con mayor tasa de error de XGBoost:')
display(stations_to_review.query("model == 'XGBoost final'").head(20))
print('Errores de XGBoost por hora:')
display(errors_by_hour.query("model == 'XGBoost final'").sort_values('hour'))


In [ ]:
# Visualizamos las horas con más errores para el modelo final.
xgb_hour_errors = errors_by_hour.query("model == 'XGBoost final'").sort_values('hour')
ax = xgb_hour_errors.plot(
    x='hour', y='tasa_error', marker='o', legend=False, figsize=(9, 4), color='#C0392B'
)
ax.set_title('Tasa de error de XGBoost por hora')
ax.set_xlabel('Hora local')
ax.set_ylabel('Tasa de error')
ax.set_xticks(range(24))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Mostramos los pares de clases más frecuentes entre los errores del modelo final.
error_pairs_xgb = (
    predictions.query("model == 'XGBoost final' and is_error")
    .groupby('error_pair', as_index=False).size()
    .rename(columns={'size': 'errores'})
    .sort_values('errores', ascending=False)
)
error_pairs_xgb.to_csv(OUTPUT_DIR / 'pares_de_error_xgboost.csv', index=False, encoding='utf-8-sig')
error_pairs_xgb


## Cómo interpretar los resultados

- Prioriza `f1_macro` y `balanced_accuracy` de `comparacion_modelo_final_y_linea_base_test.csv`.
- Las importancias identifican variables predictivas relevantes, pero no relaciones causales.
- Una estación u hora prioritaria requiere suficientes observaciones y una tasa de error elevada; conviene contrastarla con el contexto operativo antes de proponer una acción.
- Usa `pares_de_error_xgboost.csv` y `errores_por_clase_real.csv` para explicar si el modelo confunde más el vaciado, la saturación o la estabilidad.